# Burn Delta Distribution Analysis

This notebook characterizes the distribution of burn deltas in `ci_payment_details_2.csv` and proposes a fitted parameterization. The primary modeling variable is the normalized burn ratio `R = THIS_BURN / CI_BURN_RATE_MONTHS`; percent delta is then `100 * (R - 1)`.

## Why the Edge Is at -100%

For `BURN_DELTA_MONTHS_PERCENT`, the left edge for nonnegative burn is `-100%`, not `0`.

```text
R = THIS_BURN / CI_BURN_RATE_MONTHS
DeltaPct = 100 * (R - 1)
```

So zero actual burn means `R = 0`, which maps to `DeltaPct = -100%`. Negative actual burn creates values below `-100%`. Positive actual burn creates a continuous component above `-100%`.

In [ ]:
import csv
from decimal import Decimal

import numpy as np
from scipy import stats

ratio = []
delta_pct = []
with open('ci_payment_details_2.csv', newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    for row in reader:
        base = float(Decimal(row['CI_BURN_RATE_MONTHS']))
        burn = float(Decimal(row['THIS_BURN']))
        r = burn / base
        ratio.append(r)
        delta_pct.append(100 * (r - 1))

ratio = np.array(ratio)
delta_pct = np.array(delta_pct)
pos = ratio[ratio > 0]
zero_count = np.sum(ratio == 0)
negmag = -ratio[ratio < 0]

pos_params = stats.burr12.fit(pos, floc=0)
neg_params = stats.weibull_min.fit(negmag, floc=0)

print('Positive Burr XII params:', pos_params)
print('Negative magnitude Weibull params:', neg_params)
print('Weights:', {
    'negative': len(negmag) / len(ratio),
    'zero_atom': zero_count / len(ratio),
    'positive': len(pos) / len(ratio),
})

## Empirical Components

| Component | Ratio condition | Rows | Weight | Percent-delta support |
| --- | --- | --- | --- | --- |
| Negative correction component | R < 0 | 126 | 0.7667% | DeltaPct < -100% |
| Zero-burn atom / left edge | R = 0 | 109 | 0.6632% | DeltaPct = -100% |
| Positive burn continuous component | R > 0 | 16,200 | 98.5701% | DeltaPct > -100% |

## Moments and Shape

| Variable | Mean | Std dev | Skew | Excess kurtosis | Median |
| --- | --- | --- | --- | --- | --- |
| BURN_DELTA_MONTHS_PERCENT | -43.4970 | 100.8914 | 5.8809 | 64.8034 | -77.5994 |
| R = THIS_BURN / CI_BURN_RATE_MONTHS | 0.565030 | 1.008914 | 5.8809 | 64.8034 | 0.224006 |
| BURN_DELTA_MONTHS | -258,942.35 | 753,964.42 | 3.7849 | 122.6847 | -122,610.00 |

The high skew and excess kurtosis rule out a simple normal model. The absolute delta is also strongly scale-dependent, so the normalized ratio or percent delta is the better modeling target.

## Quantiles

| Probability | DeltaPct quantile | Ratio quantile |
| --- | --- | --- |
| 0.001 | -234.9233 | -1.349233 |
| 0.005 | -104.8214 | -0.048214 |
| 0.01 | -100.0000 | 0.000000 |
| 0.025 | -99.4034 | 0.005966 |
| 0.05 | -98.3254 | 0.016746 |
| 0.1 | -96.1045 | 0.038955 |
| 0.25 | -89.3992 | 0.106008 |
| 0.5 | -77.5994 | 0.224006 |
| 0.75 | -38.1378 | 0.618622 |
| 0.9 | 43.0260 | 1.430260 |
| 0.95 | 122.2578 | 2.222578 |
| 0.975 | 212.0050 | 3.120050 |
| 0.99 | 356.6667 | 4.566667 |
| 0.995 | 520.5526 | 6.205526 |
| 0.999 | 939.3046 | 10.393046 |

## Positive Component Fit

The continuous positive-burn component is `R > 0`, equivalently `DeltaPct > -100%`. Candidate positive-support distributions were fit by maximum likelihood with `loc=0`.

| Distribution | SciPy params | Log likelihood | AIC | KS statistic | Log-quantile RMSE |
| --- | --- | --- | --- | --- | --- |
| Burr XII | (1.16773589, 1.3468622, 0.0, 0.35748619) | -5,282.60 | 10,571.20 | 0.033496 | 0.137069 |
| Log-logistic / Fisk | (1.28434509, 0.0, 0.24732082) | -5,316.72 | 10,637.44 | 0.026306 | 0.235672 |
| Lognormal | (1.39765349, 0.0, 0.2424557) | -5,456.11 | 10,916.22 | 0.043276 | 0.208280 |
| Weibull | (0.76449368, 0.0, 0.47811693) | -6,006.38 | 12,016.76 | 0.073021 | 0.655699 |
| Gamma | (0.69656916, 0.0, 0.82931808) | -6,501.59 | 13,007.18 | 0.105982 | 0.710452 |

## Positive Component Quantile Check

Best AIC fit: **Burr XII**.

| Probability | Empirical positive R | Fitted positive R | Fit / empirical |
| --- | --- | --- | --- |
| 0.01 | 0.005744 | 0.005408 | 0.9415 |
| 0.05 | 0.023563 | 0.022129 | 0.9391 |
| 0.1 | 0.046047 | 0.041708 | 0.9058 |
| 0.25 | 0.109912 | 0.104610 | 0.9518 |
| 0.5 | 0.229165 | 0.254681 | 1.1113 |
| 0.75 | 0.628964 | 0.591107 | 0.9398 |
| 0.9 | 1.442433 | 1.302698 | 0.9031 |
| 0.95 | 2.236959 | 2.177249 | 0.9733 |
| 0.99 | 4.571916 | 6.494020 | 1.4204 |

## Negative Correction Component Fit

The negative component is small, so this fit should be interpreted as a correction/reversal magnitude model rather than a high-confidence physical burn distribution. Let `M = -R` for negative rows.

| Distribution | SciPy params | Log likelihood | AIC | KS statistic | Log-quantile RMSE |
| --- | --- | --- | --- | --- | --- |
| Weibull | (0.48671576, 0.0, 0.29034984) | 18.52 | -33.05 | 0.033863 | 0.342691 |
| Gamma | (0.35230439, 0.0, 1.62438797) | 18.33 | -32.65 | 0.066769 | 0.319402 |
| Burr XII | (0.48714689, 317.8125497, 0.0, 39507.20865156) | 18.50 | -31.00 | 0.034325 | 0.342917 |
| Log-logistic / Fisk | (0.67810803, 0.0, 0.11605284) | 7.39 | -10.78 | 0.071594 | 0.810611 |
| Lognormal | (2.82375094, 0.0, 0.08502822) | 0.98 | 2.04 | 0.104187 | 0.736234 |

## Proposed Distribution

Model the normalized burn ratio as a three-part mixture:

```text
R ~ p_neg * (-M) + p_zero * point_mass(0) + p_pos * X
```

with:

```text
p_neg  = 0.0076665653
p_zero = 0.0066321874
p_pos  = 0.9857012473

X ~ BurrXII(c=1.1677358941, d=1.3468621970, loc=0, scale=0.3574861947)
M ~ Weibull_min(c=0.4867157648, loc=0, scale=0.2903498365)
```

Transform to burn-delta percent with:

```text
DeltaPct = 100 * (R - 1)
```

Equivalently, in percent-delta space:

```text
with probability p_neg:  DeltaPct = -100 - 100*M
with probability p_zero: DeltaPct = -100
with probability p_pos:  DeltaPct = -100 + 100*X
```

## Plot: Empirical Distribution vs Model

The histogram is plotted in percent-delta space. The fitted density includes both continuous components; the point mass at `-100%` is shown as a vertical marker because it is an atom, not a density.



## Plot: Positive Component QQ Check

This compares empirical positive-ratio quantiles to fitted Burr XII quantiles on log scale. Deviations in the far upper tail are expected, but Burr XII captures the body and tail better than the simpler alternatives by AIC.



## Interpretation

The distribution is best described as a **left-edge-inflated, shifted heavy-tailed mixture**. The edge is `-100%` in percent-delta space because that is where zero actual burn lands. The small mass below `-100%` is a negative correction/reversal process. The large continuous component above `-100%` is positive burn, and it is heavy-tailed: most payment events are below the monthly baseline, but a meaningful tail of events is many times larger than the monthly baseline.

For modeling absolute deltas, first model `R` or `DeltaPct`, then scale by `CI_BURN_RATE_MONTHS`:

```text
BURN_DELTA_MONTHS = CI_BURN_RATE_MONTHS * (R - 1)
```

This avoids pretending that absolute deltas have one common scale across items.